In [1]:
print("test")

test


In [2]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.8 MB/s eta 0:00:00


In [3]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import medmnist
import pandas as pd
from medmnist import INFO, Evaluator

In [4]:
data_flag = 'chestmnist'
# data_flag = 'breastmnist'
download = True


info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])
# preprocessing
data_transform = transforms.Compose([
      #  transforms.Resize(224),
#cause resnet workswith 224 224  but teh images are 2828  so ths so the feautres are more meaninguly
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# load the data
from torch.utils.data import ConcatDataset

train_dataset = DataClass(split='train', transform=data_transform, download=True)
val_dataset = DataClass(split='val', transform=data_transform, download=True)
test_dataset = DataClass(split='test', transform=data_transform, download=True)

corpus = ConcatDataset([train_dataset, val_dataset, test_dataset])
X = []
Y = []

for img, label in corpus:
    X.append(img.numpy())
    Y.append(label)

X= np.array(X)
Y = np.array(Y)

100%|██████████| 82.8M/82.8M [00:05<00:00, 15.3MB/s]


In [13]:
X

array([[[[-0.52156866, -0.92941177, -0.8980392 , ..., -0.8980392 ,
          -0.8352941 , -0.827451  ],
         [-0.5764706 , -0.94509804, -0.92156863, ..., -0.8352941 ,
          -0.77254903, -0.73333335],
         [-0.4980392 , -0.81960785, -0.6392157 , ..., -0.64705884,
          -0.7882353 , -0.88235295],
         ...,
         [-0.5686275 , -0.9607843 , -0.54509807, ...,  0.73333335,
           0.6156863 ,  0.30980396],
         [-0.5529412 , -0.96862745, -0.7490196 , ...,  0.73333335,
           0.6       ,  0.14509809],
         [-0.4980392 , -0.94509804, -0.62352943, ...,  0.8039216 ,
           0.6862745 ,  0.26274514]]],


       [[[-0.9843137 , -0.9843137 , -0.9843137 , ..., -0.99215686,
          -0.99215686, -0.9607843 ],
         [-0.9843137 , -0.9843137 , -0.9843137 , ..., -0.96862745,
          -0.9843137 , -0.9607843 ],
         [-0.9843137 , -0.99215686, -0.99215686, ..., -0.654902  ,
          -0.8117647 , -0.9529412 ],
         ...,
         [-0.9843137 , -0.984313

In [43]:
X = []
Y = []

for img, label in corpus:
    X.append(img.numpy())
    Y.append(label)

X = np.array(X)
Y = np.array(Y)

## TEsting raddino for embeddings

In [32]:
corpus = ConcatDataset([train_dataset, val_dataset, test_dataset])


In [11]:
print(n_classes ,info['label'])

14 {'0': 'atelectasis', '1': 'cardiomegaly', '2': 'effusion', '3': 'infiltration', '4': 'mass', '5': 'nodule', '6': 'pneumonia', '7': 'pneumothorax', '8': 'consolidation', '9': 'edema', '10': 'emphysema', '11': 'fibrosis', '12': 'pleural', '13': 'hernia'}


In [12]:
import torch
import torchvision.models as models
from torch.utils.data import DataLoader

model = models.resnet50(weights="DEFAULT")
model.fc = torch.nn.Identity()  # remove classifier
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [28]:
from torch.utils.data import Subset

#corpus = Subset(corpus, list(range(40000)))   #no order chnage

In [29]:
print(len(corpus))

40000


In [6]:
#X = X.reshape(len(X), -1)
loader = DataLoader(
    corpus,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [35]:
features = []
labels = []
i=0
with torch.no_grad():
    for imgs, lbls in loader:
      #  print(i)
      #  i+=1
        imgs = imgs.repeat(1,3,1,1)  # convert 1-channel -> 3-channel

        feats = model(imgs)          # batch forward pass
        features.append(feats.cpu())

        labels.append(lbls)

features = torch.cat(features).numpy()
labels = torch.cat(labels).numpy()

In [36]:
print(len(features))

112120


In [37]:
saved_samples=[]
import random

def get_samples(corpus, C, size_samples=100):
    candidates = list(set(range(len(corpus))) - set(C))
    if len(candidates) <= size_samples:
        return candidates
    return random.sample(candidates, size_samples)

In [38]:
import numpy as np
import random
from sklearn.neighbors import NearestNeighbors

class GreedyCoreset:
    def __init__(self, corpus, labels=None):
        self.corpus = np.array(corpus)
        self.labels = labels
        self.n = len(corpus)
        
        # you use nn later to get the 100 nearest neighbors for each dp
        self.nn = NearestNeighbors(n_neighbors=min(100, self.n), metric='euclidean')
        self.nn.fit(self.corpus)
    
    def select(self, K=50, sample_size=100, per_label=False): #what you cocall
        if per_label and self.labels is not None:
            return self._select_per_label(K, sample_size)
        else:
            return self._select(K, sample_size)
    
    def _select(self, K, sample_size):  #the core corset selection algo 
        C = []
        
        # array to keep for each datapoint in corpus the min distance we have from a point to it
        #initialse with sth huge
        min_dists = np.full(self.n, np.inf)
        """
        min_dist is storing for each point the closest representative to it from points from C
        """
        
        # First point: pick farthest from center or random
        if K > 0:
            # Pick point farthest from mean (diverse start)
            mean = np.mean(self.corpus, axis=0)
            first_idx = np.argmax([np.linalg.norm(x - mean) for x in self.corpus])
            C.append(first_idx)
            
            # Update min distances
            for i in range(self.n):
                min_dists[i] = np.linalg.norm(self.corpus[i] - self.corpus[first_idx])
        
        while len(C) < K:
            # Sample candidates (not in C)
            """ candidates = list(set(range(self.n)) - set(C))
            if len(candidates) > sample_size:
                candidates = random.sample(candidates, sample_size)"""
            candidates=get_samples(self.corpus, C, size_samples=100)
            best_t = None
            best_utility = -np.inf
            
            for t in candidates:
                # Fast utility computation using nearest neighbors
                # Get distances from t to all points (approximate)
                dist_to_t, indexes = self.nn.kneighbors(
                    self.corpus[t].reshape(1, -1), 
                  #  n_neighbors=self.n,  #why specify n == everyone 
                    return_distance=True
                )
                dist_to_t = dist_to_t[0]
                indexes = indexes[0]

                # Compute reduction
                reduction = 0
                l_neighbors=len(dist_to_t)
                """
                amoung the k closest neighbors we picked we are gonna see 
                which of them is the cloest to the rest of datapoitns
                from the main corpus 
                """
                for j, i in enumerate(indexes): # i is dataset index  and j positiion in neigh list 
                    if i in C:
                        continue
                    if dist_to_t[j] < min_dists[i]:
                        #this bewlo is teh utility
                        # E(c) -E[c+t]
                        reduction += min_dists[i] - dist_to_t[j]
                #reduction is the "imporvment " it will bring to the corset
                #
                if reduction > best_utility:
                    best_utility = reduction
                    best_t = t
            
            if best_t is not None:
                C.append(best_t)
                
                # Update min distances
                for i in range(self.n):
                    if i in C:
                        continue
                    """
                    here you added new point t to the corset , and remember that mindist 
                    is computing dis based on points in the corset 
                    since you added new point to corset recompute to see 
                    """
                    #PP
                    dist = np.linalg.norm(self.corpus[i] - self.corpus[best_t])
                    if dist < min_dists[i]:
                        min_dists[i] = dist
            
            print(f"Selected {len(C)}/{K} tuples")
        
        return C
    
    def _select_per_label(self, K, sample_size):
        pass 
       """ unique_labels = np.unique(self.labels)  #get all of out labels here they are 
        C = []
        
        for label in unique_labels:
            label_indices = np.where(self.labels == label)[0]
            label_ratio = len(label_indices) / self.n
            label_K = max(1, int(K * label_ratio))
            
            if label_K > 0 and len(label_indices) > 0:
                # Create sub-selector for this label
                label_selector = GreedyCoreset(
                    self.corpus[label_indices], 
                    labels=None
                )
                label_C = label_selector._select(label_K, sample_size)
                
                # Map back to original indices
                C.extend(label_indices[label_C].tolist())
        
        return C
    """
    def compute_weights(self, C):
        """Compute weights for coreset points"""
        weights = np.zeros(len(C))
        
        for i in range(self.n):
            # Find closest coreset point
            min_dist = np.inf
            closest_idx = -1
            for j, c in enumerate(C):
                dist = np.linalg.norm(self.corpus[i] - self.corpus[c])
                if dist < min_dist:
                    min_dist = dist
                    closest_idx = j
            
            weights[closest_idx] += 1
        
        return weights



In [39]:
f_mini=features[:40000]
l_mini=labels[:40000]

In [40]:
selector = GreedyCoreset(f_mini, l_mini)


In [ ]:
coreset_indices = selector.select(K=1120, sample_size=200)
#weights = selector.compute_weights(coreset_indices)


In [2]:
print("test")

test


In [1]:
print(coreset_indices)

NameError: name 'coreset_indices' is not defined

In [ ]:
np.save("indices1.npz")

In [44]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

############################################
# Helper functions
############################################

def representation_error(corpus, C):
    """
    Average distance from each datapoint to its closest coreset point
    """
    total = 0
    for i in range(len(corpus)):
        d = min(np.linalg.norm(corpus[i] - corpus[c]) for c in C)
        total += d
    return total / len(corpus)


def max_distance(corpus, C):
    """
    Worst represented datapoint
    """
    max_d = 0
    for i in range(len(corpus)):
        d = min(np.linalg.norm(corpus[i] - corpus[c]) for c in C)
        if d > max_d:
            max_d = d
    return max_d


def mean_difference(corpus, C):
    """
    Difference between dataset mean and coreset mean
    """
    dataset_mean = np.mean(corpus, axis=0)
    coreset_mean = np.mean(corpus[C], axis=0)

    return np.linalg.norm(dataset_mean - coreset_mean)


def covariance_difference(corpus, C):
    """
    Difference between dataset covariance and coreset covariance
    """
    cov_dataset = np.cov(corpus.T)
    cov_coreset = np.cov(corpus[C].T)

    return np.linalg.norm(cov_dataset - cov_coreset)


############################################
# Random baseline
############################################

def random_coreset(n_points, K):
    return np.random.choice(n_points, K, replace=False)


############################################
# Evaluation
############################################

def evaluate_coreset(features, coreset_indices):

    N = len(features)
    K = len(coreset_indices)

    print("Dataset size:", N)
    print("Coreset size:", K)

    # Random subset
    random_indices = random_coreset(N, K)

    print("\nComputing metrics...")

    ########################################
    # Representation error
    ########################################

    greedy_error = representation_error(features, coreset_indices)
    random_error = representation_error(features, random_indices)

    ########################################
    # Coverage
    ########################################

    greedy_max = max_distance(features, coreset_indices)
    random_max = max_distance(features, random_indices)

    ########################################
    # Distribution similarity
    ########################################

    greedy_mean_diff = mean_difference(features, coreset_indices)
    random_mean_diff = mean_difference(features, random_indices)

    greedy_cov_diff = covariance_difference(features, coreset_indices)
    random_cov_diff = covariance_difference(features, random_indices)

    ########################################
    # Print results
    ########################################

    print("\n===== Representation Error =====")
    print("Greedy coreset :", greedy_error)
    print("Random subset  :", random_error)

    print("\n===== Max Distance (coverage) =====")
    print("Greedy coreset :", greedy_max)
    print("Random subset  :", random_max)

    print("\n===== Mean Difference =====")
    print("Greedy coreset :", greedy_mean_diff)
    print("Random subset  :", random_mean_diff)

    print("\n===== Covariance Difference =====")
    print("Greedy coreset :", greedy_cov_diff)
    print("Random subset  :", random_cov_diff)

    ########################################
    # PCA Visualization
    ########################################

    print("\nRunning PCA visualization...")

    pca = PCA(n_components=2)
    reduced = pca.fit_transform(features)

    plt.figure()

    plt.scatter(
        reduced[:,0],
        reduced[:,1],
        s=5,
        alpha=0.2,
        label="dataset"
    )

    plt.scatter(
        reduced[coreset_indices,0],
        reduced[coreset_indices,1],
        s=40,
        label="greedy coreset"
    )

    plt.scatter(
        reduced[random_indices,0],
        reduced[random_indices,1],
        s=40,
        marker="x",
        label="random subset"
    )

    plt.legend()
    plt.title("Coreset Coverage (PCA projection)")
    plt.show()

In [58]:
evaluate_coreset(f_mini, coreset_indices)

Dataset size: 40000
Coreset size: 2000

Computing metrics...


KeyboardInterrupt: 

In [57]:
print(f_mini.shape)
print(l_mini.shape)
print(mask.shape)

(40000, 2048)
(40000, 14)
(40000, 14)


In [56]:
unique, counts = np.unique(l_mini, return_counts=True)

# Keep only classes with >= 2 samples
valid_classes = unique[counts >= 2]

# Create mask
mask = np.isin(l_mini, valid_classes)

# Filter dataset
f_mini = f_mini[mask]
l_mini = l_mini[mask]

IndexError: boolean index did not match indexed array along axis 1; size of axis is 2048 but size of corresponding boolean axis is 14

###  training on corset

In [5]:
import re

with open("/kaggle/input/datasets/majdoubnourelhouda/indices-corset/indices..txt", "r") as f:
    text = f.read()

# Remove np.int64(...) wrapper
text = re.sub(r"np\.int64\((\d+)\)", r"\1", text)

# Convert string → list
indices = eval(text)

# Ensure all are ints
indices = [int(x) for x in indices]

In [7]:
len(indices)

2000

In [48]:
X_coreset = X[indices]
Y_coreset = Y[indices] 

In [49]:
print(len(X_coreset),len(Y_coreset))

2000 2000


In [50]:
print(np.sum(Y, axis=0))
print(np.sum(Y_coreset, axis=0))

[11535  2772 13307 19870  5746  6323  1353  5298  4667  2303  2516  1686
  3385   227]
[194  50 199 342  83 101  24  86  78  38  45  24  47   2]


In [21]:
Y_labels = np.argmax(Y, axis=1)

In [59]:
all_indices = np.arange(len(X))

coreset_indices = np.array(indices)

# indices NOT in coreset
remaining_indices = np.setdiff1d(all_indices, coreset_indices)

In [58]:
#coreset_indices = np.array(coreset_indices).astype(int)



X_coreset = X[indices]
Y_coreset = Y[indices]

In [60]:
np.random.seed(42)

test_200_indices = np.random.choice(
    remaining_indices,
    size=200,
    replace=False
)

In [61]:
X_test_200 = X[test_200_indices]
Y_test_200 = Y[test_200_indices]

In [62]:
print(X_coreset.shape)
print(Y_coreset.shape)
#Y_coreset = np.argmax(Y_coreset, axis=1)  #did run this but the axi 1 cased error after rerun
#print(Y_coreset.shape)


(2000, 1, 28, 28)
(2000, 14)


In [64]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class CoresetDataset(Dataset):
    def __init__(self, X, Y, transform=None):
        self.X = X
        self.Y = Y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = self.X[idx]
        label = self.Y[idx]

        img = torch.tensor(img, dtype=torch.float32)

        if img.shape[0] == 1:  # grayscale → 3 channels
            img = img.repeat(3, 1, 1)

        if self.transform:
            img = self.transform(img)

        return img, label
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ViT needs this
])
train_dataset = CoresetDataset(X_coreset, Y_coreset, transform)
test_dataset  = CoresetDataset(X_test_200, Y_test_200, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64)

In [65]:
import torchvision.models as models
import torch.nn as nn

model = models.vit_b_16(pretrained=True)

# Replace classifier
num_classes = 14
model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [66]:
for param in model.parameters():
    param.requires_grad = False

# Only train classifier
for param in model.heads.parameters():
    param.requires_grad = True

In [67]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.heads.parameters(), lr=1e-3)

def train(model, loader):
    model.train()
    total_loss = 0

    for x, y in loader:
            x = x.to(device)
            y = y.to(device).float()
        
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

    return total_loss / len(loader)

In [68]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device).float()
    
            out = model(x)
            probs = torch.sigmoid(out)
            preds = (probs > 0.5).float()
    
            correct += (preds == y).sum().item()
            total += y.numel()

    return correct / total

In [69]:
print(max(indices))

39965


In [70]:
epochs = 10

for epoch in range(epochs):
    loss = train(model, train_loader)
    acc = evaluate(model, test_loader)

    print(f"Epoch {epoch+1}: Loss={loss:.4f}, Test Acc={acc:.4f}")

Epoch 1: Loss=0.2048, Test Acc=0.9550
Epoch 2: Loss=0.1636, Test Acc=0.9539
Epoch 3: Loss=0.1604, Test Acc=0.9554
Epoch 4: Loss=0.1584, Test Acc=0.9554
Epoch 5: Loss=0.1571, Test Acc=0.9554
Epoch 6: Loss=0.1534, Test Acc=0.9554
Epoch 7: Loss=0.1529, Test Acc=0.9546
Epoch 8: Loss=0.1514, Test Acc=0.9546
Epoch 9: Loss=0.1506, Test Acc=0.9550
Epoch 10: Loss=0.1489, Test Acc=0.9554


In [44]:
task = info['task']

In [78]:
print(np.sum(Y[:40000], axis=0))         # full dataset
print(np.sum(Y_coreset, axis=0))  # coreset
print("----------")
print(np.sum(Y[:40000], axis=0)/40000 )         # full dataset
print(np.sum(Y_coreset, axis=0)/len(Y_coreset))  # coreset

[4025  989 4704 7011 2069 2271  505 1874 1640  855  881  592 1173   80]
[194  50 199 342  83 101  24  86  78  38  45  24  47   2]
----------
[0.100625 0.024725 0.1176   0.175275 0.051725 0.056775 0.012625 0.04685
 0.041    0.021375 0.022025 0.0148   0.029325 0.002   ]
[0.097  0.025  0.0995 0.171  0.0415 0.0505 0.012  0.043  0.039  0.019
 0.0225 0.012  0.0235 0.001 ]


In [ ]:
#must be true for all== final sanity check
print((labels[:40000] == Y[:40000]).all())

In [ ]:
i = indices[0] #or any random index
print(np.allclose(f_mini[i], features[i]))  # should be True